# Denoising Autoencoder on MNIST
Submission-ready notebook.

In [ ]:
import os, zipfile, random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D
from tensorflow.keras.models import Model
print(tf.__version__)

In [ ]:
ZIP_PATH='archive (2).zip'
EXTRACT_DIR='mnist_data'
if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH,'r') as z:
        z.extractall(EXTRACT_DIR)
base=os.path.join(EXTRACT_DIR,'mnist_png')
def load_split(split):
    imgs=[]
    for d in sorted(os.listdir(os.path.join(base,split))):
        folder=os.path.join(base,split,d)
        for f in os.listdir(folder):
            img=np.array(Image.open(os.path.join(folder,f)).convert('L'),dtype='float32')/255.0
            imgs.append(img)
    arr=np.array(imgs)[...,None]
    return arr
x_train=load_split('training')
x_test=load_split('testing')
print(x_train.shape,x_test.shape)

In [ ]:
noise_factor=0.5
x_train_noisy=np.clip(x_train+noise_factor*np.random.normal(size=x_train.shape),0.,1.)
x_test_noisy=np.clip(x_test+noise_factor*np.random.normal(size=x_test.shape),0.,1.)

plt.figure(figsize=(10,2))
for i in range(5):
    plt.subplot(1,5,i+1)
    plt.imshow(x_train_noisy[i].squeeze(),cmap='gray')
    plt.axis('off')
plt.show()

In [ ]:
input_img=Input(shape=(28,28,1))
x=Conv2D(32,3,activation='relu',padding='same')(input_img)
x=MaxPooling2D(2,padding='same')(x)
x=Conv2D(64,3,activation='relu',padding='same')(x)
encoded=MaxPooling2D(2,padding='same')(x)

x=Conv2D(64,3,activation='relu',padding='same')(encoded)
x=UpSampling2D(2)(x)
x=Conv2D(32,3,activation='relu',padding='same')(x)
x=UpSampling2D(2)(x)
decoded=Conv2D(1,3,activation='sigmoid',padding='same')(x)

autoencoder=Model(input_img,decoded)
autoencoder.compile(optimizer='adam',loss='binary_crossentropy')
autoencoder.summary()

In [ ]:
history=autoencoder.fit(
    x_train_noisy,x_train,
    epochs=10,
    batch_size=128,
    shuffle=True,
    validation_data=(x_test_noisy,x_test)
)

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(history.history['loss'],label='Training Loss')
plt.plot(history.history['val_loss'],label='Validation Loss')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training History')
plt.show()

In [ ]:
decoded_imgs=autoencoder.predict(x_test_noisy)

n=10
plt.figure(figsize=(18,6))
for i in range(n):
    ax=plt.subplot(3,n,i+1)
    plt.imshow(x_test[i].squeeze(),cmap='gray')
    plt.title('Original')
    plt.axis('off')

    ax=plt.subplot(3,n,i+1+n)
    plt.imshow(x_test_noisy[i].squeeze(),cmap='gray')
    plt.title('Noisy')
    plt.axis('off')

    ax=plt.subplot(3,n,i+1+2*n)
    plt.imshow(decoded_imgs[i].squeeze(),cmap='gray')
    plt.title('Denoised')
    plt.axis('off')
plt.tight_layout()
plt.show()

## Observations

- The denoising autoencoder successfully learns to reconstruct clean handwritten digits from noisy inputs.
- Binary Crossentropy loss decreases during training, indicating effective learning.
- Most Gaussian noise is removed while preserving the overall digit structure.
- Some fine details become slightly blurred due to information compression in the latent representation.
- Increasing the noise factor makes reconstruction more challenging and may require a deeper network or additional training epochs.

## Conclusion

A convolutional denoising autoencoder was implemented on the MNIST dataset. The model effectively restored noisy handwritten digit images using an encoder-decoder architecture. The results demonstrate that autoencoders are useful for image denoising and representation learning.